<a href="https://colab.research.google.com/github/GabrielMoniz/GabrielMoniz/blob/main/Many-body-simulation-training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modeling random Circuits


## Generating states as tensors

For a $n$ particle quantum state, where the $i-$th particle lives in a $d_i$ we can write a general state as
$$\psi\rangle = \sum_{d_0,...,d_n} \psi_{d_0,...,d_n} |d_0,...,d_n\rangle$$
Therefore, the state can be indentified on a $rank(n)$ tensor $\psi_{d_0,...,d_n}$, instead of only representing it as vector. there is a correspondence between the two representations. Being easier to manipulate numerically the tensor than the vector representation (for example when calculating partial trace it becomes difficult to distinguish the particles/sites, but the tensor notation make it clear)\\

The next function are for generation of a state in this notation, and there is an example of a use of numpy function that gets the index of the vectorization of the tensor by lexicographic order, connecting both notations


In [ ]:
import numpy as np

#L = number of sites
#d = dimension each site 2 for qubit
def zero_state(L, d):

  shape=(d,)*L

  # general state can be identified by a tensor |\psi> = \sum_{b_0,...,b_L-1} \psi__{b_0,...,b_L-1} |b_0,...,b_L-1>
  amplitude_tensor= np.zeros(shape, dtype=complex)

  #the state |0...0> can be identified by the tensor \psi__{0,...,0}=1 and null for all other entries
  amplitude_tensor[(0,)]=1

  return amplitude_tensor

In [ ]:
def comp_basis(L, d, bits):

  shape=(d,)*L

  # general state can be identified by a tensor |\psi> = \sum_{b_0,...,b_L-1} \psi__{b_0,...,b_L-1} |b_0,...,b_L-1>
  amplitude_tensor= np.zeros(shape, dtype=complex)

  #the state |0...0> can be identified by the tensor \psi__{0,...,0}=1 and null for all other entries
  amplitude_tensor[tuple(bits)]=1

  return amplitude_tensor

bits= (1,0,1)

comp_basis(3,2,bits).reshape(-1)

# first tuple is bits basis, the second the shape of the tensor and it return its position
# np.ravel_multi_index(bits, shape)
np.ravel_multi_index((1, 0, 1), (2, 2, 2))

np.int64(5)

## Two-qubit gate action on a state

Where it is defined a function that applies an general two-qubit gate on sites $j,j+1$ (unitary on matrix representation, but transformed into a tensor) to a state $$\psi'[\cdots,a',b',\dots] = \sum_{a,b}U_4[a',b',a,b]\;\psi[\dots,a,b,\dots].$$

And a test function with the expected expression to compare when the initial state is the computational basis $|b_0,\cdots,b_n⟩$


In [ ]:
def delta(a, b):
    return 1 if tuple(a) == tuple(b) else 0

# using np.tensordot(A, B, axes=(axes_A, axes_B))
#Here psi has $L$ axes, U4 has shape (d, d, d, d) with the index order $[a',b',a,b]$, \psi'[\dots,a',b',\dots]=\sum_{a,b}U_4[a',b',a,b]\;\psi[\dots,a,b,\dots]
# and the gate acts on qubits $j$ and $j+1$. It should return a new array and leave psi untouched.

def apply_gate(psi, U4, j):

  # \psi'[a',b',\dots] = \sum_{a,b}U_4[a',b',a,b]\;\psi[\dots,a,b,\dots]
  psi_new = np.tensordot(U4, psi, axes=([2,3], [j, j+1]))

  #moves from the first positions to the last \psi'[a',b',\dots] ->  \psi'[\dots,a'_j,b'_j,\dots]
  psi_new = np.moveaxis(psi_new, (0,1), (j,j+1))

  return psi_new



def test(L,d, bits, j):

  shape=(d,)*L

  psi_test = np.zeros(shape, dtype=complex)

  for b_list in np.ndindex(shape):
    psi_test[b_list] = U4[b_list[j], b_list[j+1], bits[j], bits[j+1]] * delta(b_list[:j], bits[:j]) * delta(b_list[j+2:] ,bits[j+2:])

  return psi_test



L=4
d=3

rng = np.random.default_rng(0)                    # random generator, fixed seed
Z = rng.normal(size=(d**2, d**2)) + 1j * rng.normal(size=(d**2, d**2))
Q, R = np.linalg.qr(Z)                            # Z = Q R, with Q unitary
U4 = Q.reshape(d, d, d, d)



for bits in np.ndindex((d,) * L):
    for j in range(L - 1):
        assert np.allclose(apply_gate(comp_basis(L, d, bits), U4, j),
                           test(L, d, bits, j))

## Haar Distribution

THe Haar-Distribution is a notion of uniform distribution on the space of Unitary matrices, such that it have no prefered direction on $\mathbb C^n$. If $U$ is Haar-distributed on $\mathrm U(n)$ and $V$ is a
fixed unitary, then $VU$ has the same distribution as $U$


Let $U$ be Haar-distributed on $\mathrm U(n)$, $X$ a fixed $n\times n$ matrix, its definition imposes
$$\mathbb E[UXU^{\dagger}]=\frac{\text{Tr}(X)}{n} \mathbf{1}$$
Removing any information from the operators besides its trace.



### Methods

Our method to generate the unitary influences the probability distribution. A method is a pair: a random input $X$ with a known distribution and a map $f$ into $U(n)$. The distribution of $U=f(X)$ then follows from $P(U\in S)=P(X\in f^{-1}(S))$. We want $U$ to be Haar-distributed, i.e. $VU$ has the same distribution as $U$ for every fixed unitary $V$ (this distribution is unique, Watrous, *The Theory of Quantum Information*, Thm 7.21).

**Lie Algebra**

One can parametrize a unitary by its Lie algebra, $$U = e^{i \sum_k z_k H_k},$$ for $(H_k)_{k=1,\cdots,n^2}$ a basis of the space of $n\times n$ Hermitian matrices. Sampling the variables $z_k$ uniformly in $[-\pi,\pi]$ does not lead to a Haar distribution.

The natural argument for invariance fails, because multiplying by a unitary is not a translation of the $z_k$. For $z_k$ random variables and $\alpha_k$ constants
$$V U = e^{i \sum_k \alpha_k H_k} e^{i \sum_k z_k H_k} \neq e^{i \sum_k (\alpha_k+z_k) H_k}$$ in general, because it is not possible to construct a basis with only commuting elements: for $n\geq 2$ not all Hermitian matrices commute (e.g. Pauli $X$ and $Z$), while linear combinations of commuting matrices would all commute. So you cannot guarantee invariance this way.

The actual failure comes from the Jacobian of $z\mapsto U$. Restrict to $H=\sum_k z_k H_k$ with eigenvalues $\lambda_1,\dots,\lambda_n\in(-\pi,\pi]$, where $H\mapsto U$ is one-to-one. Then $U$ is Haar if and only if $z$ has the joint density
$$p(z)\propto\prod_{a<b}\left[\frac{\sin\big((\lambda_a-\lambda_b)/2\big)}{(\lambda_a-\lambda_b)/2}\right]^2.$$ This density is not constant, so uniform $z$ is not Haar (on the box $[-\pi,\pi]^{n^2}$ the map is not one-to-one, and the failure is shown numerically below). It is also not a product of single-coefficient densities, so the $z_k$ cannot be sampled independently in any basis; changing the basis only changes the normalization.

**QR decomposition**

Start from a random matrix $A$ with independent complex Gaussian entries. Its density is $\propto e^{-\frac12\mathrm{Tr}(A^\dagger A)}$, and $A\to VA$ preserves both $\mathrm{Tr}(A^\dagger A)$ and the volume element, so $VA$ has the same distribution as $A$. Hence, if a rule $g: \mathbb C^{n \times n} \rightarrow U(n)$ satisfies $g(VA)=V\,g(A)$, then $U=g(A)$ and $VU=g(VA)$ have the same distribution, i.e. $U$ is Haar.

For the QR decomposition you could in principle produce an invariant distribution, since $A=QR \Rightarrow VA = (VQ)R$, where $VQ$ would be the new unitary in a QR decomposition of $VA$. But this is not a unique choice, since $A=QR=(QD)(D^\dagger R)$ for every diagonal unitary $D=\text{diag}(e^{i\phi_1},\cdots,e^{i\phi_n})$. Consider the unique decomposition $A=Q_+R_+$ in which $R_+$ has a positive diagonal (unique for invertible $A$). The python function `np.linalg.qr` returns $Q(A)=Q_+D_A$ and $R(A)=D_A^\dagger R_+$ for $$D_A=\mathrm{diag}(s_0,\dots,s_{n-1}),\qquad s_k=-\,\mathrm{sign}(\mathrm{Re}\,\alpha_k)\in\{+1,-1\},$$ where $\alpha_k$ is the $(k,k)$ entry at step $k$ of the reduction. In particular $\alpha_0=A_{00}$, which changes under $A\to VA$, whereas $R_+$ does not. So $D_{VA}\neq D_A$ and $Q(VA)\neq V\,Q(A)$ in general.


**Polar decomposition**

The polar decomposition $A = UP$ is invariant. To see this notice that $P= (A^\dagger A)^{1/2}$, so $A \to VA$ makes $P \to (A^\dagger V^\dagger VA)^{1/2}=(A^\dagger A)^{1/2}$, i.e. $P$ is the same for $A$ and $VA$. Since $A$ is invertible with probability 1, $P$ is invertible and $U=AP^{-1}$ is uniquely determined by $A$: there is no library choice to be made. Hence $U(VA)=VAP^{-1}=V\,U(A)$, and the decomposition satisfies $VA = (VU)P$. Numerically, from the SVD $A=W\Sigma K$ one gets $U=WK$ (`W, s, Vh = np.linalg.svd(A); U = W @ Vh`).

**Choice.** Both the polar decomposition and the QR decomposition with positive diagonal produce Haar unitaries. We use the polar decomposition, since its unitary factor is unique and no convention has to be fixed.

In [ ]:
def haar_unitary(n, rng):
  rng = np.random.default_rng(0)                    # random generator, fixed seed
  Z = rng.normal(size=(d**2, d**2)) + 1j * rng.normal(size=(d**2, d**2))
